# Automated Essay Scoring (AES) dengan Finetuning LoRA

Notebook ini berfokus pada implementasi penilaian esai otomatis menggunakan model bahasa lokal **TinyLlama-1.1B**. Kita akan melakukan proses *Finetuning* berbasis **Parameter-Efficient Fine-Tuning (PEFT)** dengan metode **LoRA (Low-Rank Adaptation)**.

Di akhir proses training, kita akan melakukan evaluasi performa model pada 20 sampel data untuk membandingkan kecocokan antara skor asli dari guru dengan skor hasil prediksi model LoRA menggunakan metrik **Quadratic Weighted Kappa (QWK)**.

## 1. Penginstalan Library dan Persiapan Lingkungan

Langkah pertama adalah menginstal pustaka utama Hugging Face seperti `transformers`, `peft` untuk pengelolaan modul LoRA, serta `accelerate` dan `datasets` untuk efisiensi pemrosesan data.

In [ ]:
!pip install transformers==4.41.2
!pip install peft==0.10.0
!pip install accelerate
!pip install datasets
!pip install sentencepiece

## 2. Autentikasi dan Manajemen Memori GPU

Melakukan masuk log (login) ke Hugging Face Hub serta mengosongkan cache CUDA agar alokasi VRAM kartu grafis T4 di Google Colab bersih dan optimal sebelum memuat model.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch
torch.cuda.empty_cache()

## 3. Pemuatan Dataset dan Pengambilan Sampel Eksperimen

Kita mengunggah file data `train.csv`, menyaring kolom teks esai beserta target skor, membersihkan data yang kosong, lalu mengambil 20 sampel data secara acak untuk kebutuhan pengujian cepat.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")
df.head()

In [ ]:
print(df.columns)

In [ ]:
df = df[['full_text', 'score']]
df = df.dropna()
df = df.sample(20, random_state=42)

## 4. Pembuatan Prompt Berbasis Instruksi

Menyusun fungsi pembuat prompt untuk membungkus teks esai ke dalam format peran instruksional yang jelas bagi LLM.

In [ ]:
def make_prompt(row):
    prompt = f"""
You are an expert essay grader.

Essay:
{row['full_text']}

Give a score from 1 to 6 only.
"""

    return {
        "text": prompt,
        "label": int(row['score'])
    }

data = df.apply(make_prompt, axis=1).tolist()

## 5. Pemuatan Pre-trained Model dan Tokenizer

Mengunduh arsitektur dasar model **TinyLlama-1.1B-Chat** beserta pengondisian token padding-nya.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

## 6. Konfigurasi Pembobotan LoRA (PEFT)

Mengatur parameter adaptasi peringkat rendah (*Low-Rank Adaptation*) pada target modul proyeksi perhatian model (`q_proj`, `v_proj`).

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

## 7. Tokenisasi Dataset

Mengonversi deretan teks instruksi dan pasangan label target menjadi representasi ID token tensor numerik yang siap dikonsumsi oleh model saat pelatihan.

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(data)

def tokenize(example):
    text = example["text"] + " Score: " + str(example["label"])

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize)
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

## 8. Inisialisasi Parameter dan Proses Training Model

Menentukan konfigurasi *hyperparameters* untuk `TrainingArguments` lalu mengeksekusi fungsi latih model bawaan Hugging Face.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    report_to="notebook",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=50,
    learning_rate=2e-4
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("aes-lora-model")
tokenizer.save_pretrained("aes-lora-model")

## 9. Eksperimen Evaluasi Kuantitatif dan Perhitungan Metrik QWK

Pada bagian penutup ini, kita meniru alur pengujian komparatif terstruktur. Kita akan menguji ke-20 sampel esai menggunakan bar kemajuan (`tqdm`), mengekstraksi skor prediksi angka menggunakan regular expression (`re`), menampilkan ringkasan tabel komparatif, dan menghitung pencapaian metrik **Quadratic Weighted Kappa (QWK)**.

In [ ]:
import re
from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score
from IPython.display import display, Markdown

df_eksperimen = df.copy()
prediksi_lora = []

print("Memulai penilaian model LoRA pada 20 esai...")

for index, row in tqdm(df_eksperimen.iterrows(), total=df_eksperimen.shape[0]):
    esai_target = row['full_text']
    
    prompt_eval = f"""
You are an expert essay grader.

Essay:
{esai_target}

Give a score from 1 to 6 only.
"""
    
    inputs = tokenizer(prompt_eval, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            temperature=0.0,
            do_sample=False
        )
    
    hasil_teks = tokenizer.decode(outputs[0], skip_special_tokens=True)
    respon_saja = hasil_teks[len(prompt_eval):]
    
    match = re.search(r'[1-6]', respon_saja)
    skor = int(match.group()) if match else 3
    prediksi_lora.append(skor)

df_eksperimen['lora_score'] = prediksi_lora
print("\nPenilaian Evaluasi LoRA Selesai!\n")

# Mengonversi tabel dataframe ke format Markdown string
# Format ini 100% aman dan didukung penuh oleh parser preview GitHub
markdown_table = df_eksperimen[['score', 'lora_score']].to_markdown()
display(Markdown(markdown_table))

qwk_lora = cohen_kappa_score(
    df_eksperimen['score'].astype(int),
    df_eksperimen['lora_score'],
    weights='quadratic'
)
print(f"\n=== HASIL EVALUASI METRIK PERFORMANCE ===")
print(f"Skor QWK LoRA (20 Sampel): {qwk_lora:.4f}")